In [45]:
import requests

In [46]:
api = "https://data.gov.au/data/api/3/action/package_search?q=austender"


In [47]:
response = requests.get(api)

In [48]:
if response.status_code == 200:
  print("Connection is successful and the API is working")
else:
  print("Connection is unsuccessful and the API is not working", response.status_code)

Connection is successful and the API is working


In [49]:
if response.status_code == 200:
  # convert the reponse to json
  data = response.json()

  # printing data to understand the structure
  print(data)


{'help': 'https://data.gov.au/data/api/3/action/help_show?name=package_search', 'success': True, 'result': {'count': 9, 'facets': {}, 'results': [{'author': 'Department of Finance and Deregulation', 'author_email': None, 'contact_point': 'data.gov@finance.gov.au', 'creator_user_id': '91af4eef-efb1-4e4e-87f2-171d81977ae0', 'duplicate_score': 1, 'geospatial_topic': [], 'id': 'ab94e84b-114a-4b57-b497-725ad4ca975e', 'isopen': False, 'license_id': 'cc-by', 'license_title': 'Creative Commons Attribution 3.0 Australia', 'license_url': 'http://creativecommons.org/licenses/by/3.0/au/', 'maintainer': 'Department of Finance and Deregulation', 'maintainer_email': None, 'metadata_created': '2013-05-12T10:24:43.771198', 'metadata_modified': '2023-08-11T02:09:30.523037', 'name': 'austender-contract-notice-export', 'notes': '    \n\nThe Contract Notice Export is an export file containing contract notice data\nfor all Australian Government agencies who report procurement on AusTender\npublished over a 

In [50]:
#extract the count from the results

total_datasets = data['result']['count']
print("Total AusTender datasets found:", total_datasets)

Total AusTender datasets found: 9


In [51]:
first_dataset = data['result']['results'][0]

resource_id = first_dataset['resources'][0]['id']

print("Found Resource ID for the data table:", resource_id)

data_url = f"https://data.gov.au/data/api/3/action/datastore_search?resource_id={resource_id}&limit=5"

data_response = requests.get(data_url)

if data_response.status_code == 200:
  contract_data = data_response.json()

  # the actual row of data are stored inside 'result' -> 'records'
  records = contract_data['result']['records']

  print(f"\nSuccessfully pulled {len(records)} contracts!")

  #lets print out the first contracts to see what columns we have to work with

  print("\n   First contract details    ")
  first_contract = records[0]

  print("Supplier Name:", first_contract.get('Supplier Name', 'Not Found'))
  print("ABN:", first_contract.get('Supplier ABN', 'Not Found'))
  print("Value (AUD):", first_contract.get('Value', 'Not Found'))
  print("Agency:", first_contract.get('Agency', 'Not Found'))
else:
  print("Failed to get contract data. Error:", data_response.status_code)


Found Resource ID for the data table: e0c6d7b0-1043-498f-8b49-4cb695b49d99
Failed to get contract data. Error: 302


In [52]:
import pandas as pd


first_dataset = data['result']['results'][0]


download_url = first_dataset['resources'][0]['url']
print("Found Direct File URL:", download_url)

print("Downloading and reading Excel data... (this might take a few seconds)")




try:
    df = pd.read_excel(download_url, nrows=100)
    print(f"\nSuccessfully loaded {len(df)} contracts using Excel parser!")
except Exception as e:
    print(f"\nExcel parser failed. It might be a fake .xls file (actually a CSV). Trying CSV parser with Windows encoding...")
    df = pd.read_csv(download_url, encoding='cp1252', nrows=100, sep='\t') # Often they use tabs
    print(f"\nSuccessfully loaded {len(df)} contracts using Text parser!")


print("\n--- Available Columns ---")
print(df.columns.tolist())


print("\n--- Sample Data ---")
columns_to_view = ['Supplier Name', 'Supplier ABN', 'Value']

existing_columns = [col for col in columns_to_view if col in df.columns]
print(df[existing_columns].head(3))

Found Direct File URL: http://data.gov.au/storage/f/2013-05-12T202412/tmph1yFnyAusTender-Contract-Notice-Export_130105.xls

Excel parser failed. It might be a fake .xls file (actually a CSV). Trying CSV parser with Windows encoding...

Successfully loaded 100 contracts using Text parser!

--- Available Columns ---
['Contract Notice Export']

--- Sample Data ---
Empty DataFrame
Columns: []
Index: [(Agency, Parent CN ID, CN ID, Publish Date, Amendment Date, Status, StartDate, EndDate, Value, Description, Agency Ref. ID, Category, Procurement Method, ATM ID, SON ID, Confidentiality - Contract, Confidentiality - Contract Reason(s), Confidentiality - Outputs, Confidentiality - Outputs Reason(s), Consultancy, Consultancy Reason(s), Amendment Reason, Supplier Name, Supplier Address, Supplier City, Supplier Postcode, Supplier Country, Supplier ABNExempt, Supplier ABN, Agency Branch, Agency Divison, Agency Postcode), (="Defence Materiel Organisation", ="", ="CN1121942", 04-Jan-13 07:29 AM,  , =

In [53]:
import urllib.request

download_url = "http://data.gov.au/storage/f/2013-05-12T202412/tmph1yFnyAusTender-Contract-Notice-Export_130105.xls"

print("Downloading raw file to inspect headers...\n")

# Request the file
req = urllib.request.Request(download_url, headers={'User-Agent': 'Mozilla/5.0'})
response = urllib.request.urlopen(req)

# Read and decode the first 15 lines
print("--- RAW FILE TOP 15 LINES ---")
for i in range(15):
    # Decode using cp1252 to handle Windows encoding
    raw_line = response.readline().decode('cp1252').strip()
    print(f"Row {i}: {raw_line}")


--- RAW FILE TOP 15 LINES ---
Row 0: Contract Notice Export
Row 1: 
Row 2: Agency	Parent CN ID	CN ID	Publish Date	Amendment Date	Status	StartDate	EndDate	Value	Description	Agency Ref. ID	Category	Procurement Method	ATM ID	SON ID	Confidentiality - Contract	Confidentiality - Contract Reason(s)	Confidentiality - Outputs	Confidentiality - Outputs Reason(s)	Consultancy	Consultancy Reason(s)	Amendment Reason	Supplier Name	Supplier Address	Supplier City	Supplier Postcode	Supplier Country	Supplier ABNExempt	Supplier ABN	Agency Branch	Agency Divison	Agency Postcode
Row 3: ="Defence Materiel Organisation"	=""	="CN1121942"	04-Jan-13 07:29 AM	 	="published"	18-Dec-11	28-Jan-13	11731.17	" Motor Windscreen Wiper, Electric. C/W Crank & Plate Assembly, 9.64inch Nom O/A Long, 3.88inch Nom O/A Depth. Single Line Purchase Order. Part No: E-005-280. "	="DNLMN1"	="Fire fighting equipment"	="Limited tender"	=""		="No"	=""	="No"	=""	="No"	=""	""	="Speedscreen Pty Ltd"	="PO Box 273"	="Flemington"	="3031"	="A

In [54]:
import pandas as pd

download_url = "http://data.gov.au/storage/f/2013-05-12T202412/tmph1yFnyAusTender-Contract-Notice-Export_130105.xls"

print("Downloading and cleaning AusTender data...")

# Skip the title and blank row (skiprows=2)
# Force tab-separation (sep='\t')
df = pd.read_csv(download_url, encoding='cp1252', sep='\t', skiprows=2, nrows=100)

# Clean the weird Excel text formatting (="Value")
df = df.replace({'=': '', '"': ''}, regex=True)

print("\n--- Available Columns ---")
print(df.columns.tolist())

# View our target columns
columns_to_view = ['Supplier Name', 'Supplier ABN', 'Value', 'Agency']
existing_columns = [col for col in columns_to_view if col in df.columns]

print("\n--- Cleaned Sample Data ---")
print(df[existing_columns].head(3))


--- Available Columns ---
['Agency', 'Parent CN ID', 'CN ID', 'Publish Date', 'Amendment Date', 'Status', 'StartDate', 'EndDate', 'Value', 'Description', 'Agency Ref. ID', 'Category', 'Procurement Method', 'ATM ID', 'SON ID', 'Confidentiality - Contract', 'Confidentiality - Contract Reason(s)', 'Confidentiality - Outputs', 'Confidentiality - Outputs Reason(s)', 'Consultancy', 'Consultancy Reason(s)', 'Amendment Reason', 'Supplier Name', 'Supplier Address', 'Supplier City', 'Supplier Postcode', 'Supplier Country', 'Supplier ABNExempt', 'Supplier ABN', 'Agency Branch', 'Agency Divison', 'Agency Postcode', 'Unnamed: 32']

--- Cleaned Sample Data ---
                  Supplier Name Supplier ABN        Value  \
0           Speedscreen Pty Ltd  31004239865     11731.17   
1  REDBACK BOOT COMPANY PTY LTD  38003845805  25022000.00   
2         Fire Response Pty Ltd  32003635103     29546.00   

                          Agency  
0  Defence Materiel Organisation  
1  Defence Materiel Organisat

In [55]:
# Save the cleaned data to a CSV file
df.to_csv('austender_sme_contracts.csv', index=False)
print("Data successfully saved to austender_sme_contracts.csv!")

Data successfully saved to austender_sme_contracts.csv!


In [57]:
from google.colab import files
# files.download('austender_sme_contracts.csv')